# Continuum airglow at fixed geometry

This example uses the public NSB Python API to vary explicit F10.7 solar radio
flux at one fixed Paranal observer, ICRS direction and UTC instant. It integrates
the PALACE continuum over 300–650 nm and produces one figure. No external data
are required: NSB uses its bundled, manifest-verified scientific assets.

This example targets the v0.2.0 public API with the explicit
PALACE continuum model. For a publication, record the exact released version
or source commit and environment. This demonstration is not an absolute
observational validation.

Build in an isolated environment (Linux/macOS commands; on Windows use the
venv's `Scripts/python.exe`). From a separate directory once v0.2.0 is published (for pre-release testing,
replace the tag with the full reviewed commit SHA):

```bash
git clone https://github.com/VPRamon/NSB.git nsb-example-source
cd nsb-example-source
git checkout --detach v0.2.0
python3 -m venv .example-venv
.example-venv/bin/python -m pip install maturin==1.15.0 matplotlib==3.10.8
.example-venv/bin/python -m maturin build --release --locked --out target/example-wheels
.example-venv/bin/python -m pip install target/example-wheels/nsb_py-*.whl
```

Open this notebook with that environment as the Python kernel (install
`ipykernel` in the venv if your notebook editor requires it). Run all cells.
Rust 1.89+ and a native build toolchain are required. Siderust and Tempoch are installed from their published releases.
Initial dependency acquisition requires network access; the calculation itself performs no downloads. Nothing
installs packages or writes result files when a code cell is executed.


In [ ]:
from datetime import datetime, timezone
import matplotlib.pyplot as plt
import nsb
import siderust

observer = siderust.Observer(-70.4045, -24.6272, 2635.0)  # east-positive degrees, metres
query = nsb.PointQuery(
    observer, datetime(2023, 9, 15, 5, tzinfo=timezone.utc),
    siderust.Direction(10.0, -20.0),  # ICRS RA, Dec in degrees
    components=nsb.ComponentMask.AIRGLOW,
)
solar_flux = [67.0, 80.0, 100.0, 120.0, 140.0, 166.0]  # sfu
radiance = []
for sfu in solar_flux:
    config = (nsb.NsbModelConfig.generic_clear_sky()
              .with_airglow_model(nsb.AirglowModel.PARANAL_PALACE_V1)
              .with_solar_radio_flux_sfu(sfu))
    evaluator = nsb.NsbEvaluator(config)
    radiance.append(evaluator.integrate_band(query, 300.0, 650.0))
assert all(b > a > 0 for a, b in zip(radiance, radiance[1:]))


In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4.2), layout="constrained")
ax.plot(solar_flux, radiance, "o-")
ax.set(xlabel="F10.7 solar radio flux (sfu)",
       ylabel="Photon radiance (photons cm⁻² ns⁻¹ sr⁻¹)",
       title="PALACE continuum, 300–650 nm, fixed Paranal geometry")
ax.grid(alpha=0.2)
plt.show()


## Interpretation

The ordinate is band-integrated photon radiance, not detector counts or
photometric magnitude. The explicit solar inputs span 67–166 sfu; all other
inputs stay fixed. The continuum rises for this configuration. This planning
climatology is derived from Paranal and does not certify absolute accuracy at
other sites or on an individual night. Oxygen lines, starlight, zodiacal light
and moonlight are excluded; unknown observational uncertainty is not zero.

The figure demonstrates public API use and deterministic parameter control.
For scope and evidence, consult the
[scientific validation guide](../docs/scientific-validation.md), NSB's
[airglow guide](../docs/nsb_components/airglow/README.md) and
[restricted-band contract](../docs/specifications/restricted-band-radiance.md)
for model provenance and domain limits.
